## Loading the data and packages

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from datetime import timedelta
import os

In [2]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
df = pd.read_csv(os.path.join(DATA_DIR, 'comments_labeled.csv'), parse_dates=['comment_date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 182552 entries, 0 to 182551
Data columns (total 28 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Unnamed: 0                       182552 non-null  int64         
 1   comment_id                       182552 non-null  str           
 2   parent_id                        40093 non-null   str           
 3   parent_text                      40092 non-null   str           
 4   text_preprocessed                182552 non-null  str           
 5   video_title                      182552 non-null  str           
 6   video_context                    170928 non-null  str           
 7   video_id                         182552 non-null  str           
 8   comment_date                     182552 non-null  datetime64[us]
 9   author_hash                      182552 non-null  str           
 10  days_from_genai_announcement     182552 non-null  int64

## Dashboard

In [5]:
# =============================================================================
# INTERACTIVE DASHBOARD — ipywidgets
# Drop this as a single cell at the end of the notebook.
# Requires: df already loaded and processed (sentiment_score, weight_factor,
#           weighted_sentiment columns)
# =============================================================================

# ── Constants ────────────────────────────────────────────────────────────────

AI_CUTOFF = pd.Timestamp('2025-11-12')   # Hard cutoff for AI_Voice_Related

# Known events — used directly as anchors regardless of volume spike
KNOWN_EVENTS = {
    '2021-12-09': 'Game Announcement',
    '2023-05-01': 'PvPvE Design Pivot',
    '2025-06-06': 'Business Model Change',
    '2025-10-30': 'Official Game Launch',
    '2025-11-12': 'AI Voice Acting News',
    '2026-01-23': 'Roadmap 2026',
    '2026-03-15': 'AI Voice Replacement',
}

CATEGORY_OPTIONS = [
    ('Game Related',           'Game_Related'),
    ('AI Voice Controversy',   'AI_Voice_Related'),
    ('Business Model',         'Business_Model_Related'),
    ('Non-Game (Others)',      'Others'),
]

PRESET_RANGES = {
    '1 Month':  1,
    '3 Months': 3,
    '6 Months': 6,
    '1 Year':   12,
    'All Time': None,
}

CATEGORY_COLORS = {
    'Game_Related':            '#2980B9',
    'AI_Voice_Related':        '#8E44AD',
    'Business_Model_Related':  '#E67E22',
    'Others':                  '#7F8C8D',
}

SENTIMENT_COLORS = {
    'Pos':        '#2ecc71',
    'Neu':        '#95a5a6',
    'Neg':        '#e74c3c',
    'Ambivalent': '#f39c12',
}

# ── Helper functions ──────────────────────────────────────────────────────────

def _filter_df(category, months):
    """Return a filtered slice of df for the chosen category + time range."""
    # Category filter
    subset = df[df['category'] == category].copy()

    # AI hard cutoff
    if category == 'AI_Voice_Related':
        subset = subset[subset['comment_date'] >= AI_CUTOFF]

    # Time range filter (relative to the last date in the filtered subset)
    if months is not None and not subset.empty:
        end_date   = subset['comment_date'].max()
        start_date = end_date - pd.DateOffset(months=months)
        subset     = subset[subset['comment_date'] >= start_date]

    return subset


def _get_daily(subset, rolling=14):
    """Aggregate to daily stats with rolling MA columns."""
    if subset.empty:
        return pd.DataFrame()

    daily = subset.groupby(subset['comment_date'].dt.date).agg(
        volume                  = ('comment_id',          'count'),
        raw_sentiment           = ('sentiment_score',     'mean'),
        sentiment_std           = ('sentiment_score',     'std'),
        sum_weighted_sentiment  = ('weighted_sentiment',  'sum'),
        sum_weights             = ('weight_factor',       'sum'),
    ).reset_index()

    daily['comment_date']       = pd.to_datetime(daily['comment_date'])
    daily['weighted_sentiment'] = (daily['sum_weighted_sentiment']
                                   / daily['sum_weights'])
    daily['raw_ma']             = (daily['raw_sentiment']
                                   .rolling(rolling, min_periods=1).mean())
    daily['weighted_ma']        = (daily['weighted_sentiment']
                                   .rolling(rolling, min_periods=1).mean())
    return daily


def _decay_function(t, s0, lam):
    return s0 * np.exp(-lam * t)


def _fit_decay(daily, baseline_window=30):
    """Dynamic decay fit (same logic as CHANGE 1)."""
    if daily.empty or len(daily) < 5:
        return None

    peak_idx  = daily['volume'].idxmax()
    peak_date = daily.loc[peak_idx, 'comment_date']

    pre_peak     = daily[daily['comment_date'] < peak_date].tail(baseline_window)
    baseline_vol = (pre_peak['volume'].mean() if not pre_peak.empty
                    else daily['volume'].mean() * 0.3)

    df_post         = daily[daily['comment_date'] >= peak_date].copy()
    df_post['days'] = (df_post['comment_date'] - peak_date).dt.days

    returned    = df_post[df_post['volume'] <= baseline_vol]
    cutoff_days = int(returned.iloc[0]['days']) if not returned.empty else 120
    df_fit      = df_post[df_post['days'] <= cutoff_days]

    x_data = df_fit['days'].values
    y_data = df_fit['volume'].values

    try:
        popt, _ = curve_fit(_decay_function, x_data, y_data,
                            p0=[y_data[0], 0.1], maxfev=5000)
        s0_fit, lam_fit = popt
        half_life = np.log(2) / lam_fit
        fit_dates = [peak_date + timedelta(days=int(d)) for d in x_data]
        fit_vals  = _decay_function(x_data, *popt)
        return {
            'peak_date':    peak_date,
            'half_life':    half_life,
            'lam':          lam_fit,
            'fit_dates':    fit_dates,
            'fit_values':   fit_vals,
            'baseline_vol': baseline_vol,
            'fit_days':     cutoff_days,
        }
    except Exception:
        return None


# ── Plotting ──────────────────────────────────────────────────────────────────

def _draw_dashboard(category, time_label):
    months  = PRESET_RANGES[time_label]
    subset  = _filter_df(category, months)
    cat_col = CATEGORY_COLORS[category]

    fig = plt.figure(figsize=(16, 14))
    fig.patch.set_facecolor('#F8F9FA')
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    cat_label = {v: k for k, v in CATEGORY_OPTIONS}[category]
    fig.suptitle(
        f'Arc Raiders — {cat_label}   ·   {time_label}',
        fontsize=15, fontweight='bold', y=0.98,
    )

    # ── Guard: no data ────────────────────────────────────────────────────────
    if subset.empty:
        ax = fig.add_subplot(gs[:, :])
        ax.text(0.5, 0.5, 'No data for this selection.',
                ha='center', va='center', fontsize=14, color='grey')
        ax.axis('off')
        plt.show()
        return

    daily  = _get_daily(subset)
    decay  = _fit_decay(daily)

    # ── Panel 1 (top, full width): Volume + Sentiment timeline ───────────────
    ax1   = fig.add_subplot(gs[0, :])
    ax1_s = ax1.twinx()

    ax1.bar(daily['comment_date'], daily['volume'],
            color=cat_col, alpha=0.20, width=1, label='Daily Volume')
    ax1_s.plot(daily['comment_date'], daily['raw_ma'],
               color='#95a5a6', linewidth=1.5, linestyle='--',
               label='Raw Sentiment (14d MA)', alpha=0.9)
    ax1_s.plot(daily['comment_date'], daily['weighted_ma'],
               color=cat_col, linewidth=2.2,
               label='Weighted Sentiment (14d MA)')
    ax1_s.fill_between(
        daily['comment_date'],
        daily['weighted_ma'] - 0.5 * daily['sentiment_std'].fillna(0).rolling(14, min_periods=1).mean(),
        daily['weighted_ma'] + 0.5 * daily['sentiment_std'].fillna(0).rolling(14, min_periods=1).mean(),
        color=cat_col, alpha=0.08, label='±½σ Polarization Band',
    )
    ax1_s.axhline(0, color='black', linewidth=0.7, linestyle=':', alpha=0.4)
    ax1_s.set_ylim(-1.2, 1.2)

    # AI cutoff marker
    if category == 'AI_Voice_Related':
        ax1.axvline(AI_CUTOFF, color='#E74C3C', linewidth=1.5,
                    linestyle=':', label='AI Announcement (Nov 12, 2025)')

    # Known events inside the visible window
    date_min = daily['comment_date'].min()
    date_max = daily['comment_date'].max()
    for edate_str, elabel in KNOWN_EVENTS.items():
        edate = pd.Timestamp(edate_str)
        if date_min <= edate <= date_max:
            ax1.axvline(edate, color='#333', linewidth=1,
                        linestyle=':', alpha=0.5)
            ax1.text(edate, daily['volume'].max() * 1.02,
                     elabel.replace(' ', '\n'),
                     fontsize=5.5, color='#555', rotation=40,
                     ha='left', va='bottom')

    ax1.set_title('A.  Volume & Sentiment Timeline', loc='left',
                  fontsize=11, fontweight='bold')
    ax1.set_ylabel('Comment Volume', fontsize=10)
    ax1_s.set_ylabel('Sentiment Score', color=cat_col, fontsize=10)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1_s.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=7, loc='upper left', framealpha=0.85)

    # ── Panel 2 (middle-left): Sentiment Distribution ────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])

    sent_counts = subset['sentiment'].value_counts()
    sent_order  = [s for s in ['Pos', 'Neu', 'Neg', 'Ambivalent']
                   if s in sent_counts.index]
    colors      = [SENTIMENT_COLORS[s] for s in sent_order]
    bars        = ax2.bar(sent_order, sent_counts[sent_order],
                          color=colors, edgecolor='white', linewidth=0.8)

    total = sent_counts.sum()
    for bar, val in zip(bars, sent_counts[sent_order]):
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + total * 0.01,
                 f'{val:,}\n({100*val/total:.1f}%)',
                 ha='center', va='bottom', fontsize=8)

    ax2.set_title('B.  Sentiment Distribution', loc='left',
                  fontsize=11, fontweight='bold')
    ax2.set_ylabel('Comment Count', fontsize=10)
    ax2.set_xlabel('Sentiment Label', fontsize=10)

    # Stats annotation
    mean_score = subset['sentiment_score'].mean()
    ax2.text(0.97, 0.97,
             f'n = {total:,}\nMean score = {mean_score:+.3f}',
             transform=ax2.transAxes, fontsize=8,
             va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                       edgecolor='#ccc', alpha=0.9))

    # ── Panel 3 (middle-right): Weighted vs Raw Sentiment Scatter ────────────
    ax3 = fig.add_subplot(gs[1, 1])

    # Sample for readability (max 3 000 points)
    sample = subset.sample(min(3000, len(subset)), random_state=42)
    ax3.scatter(sample['sentiment_score'],
                sample['weighted_sentiment'],
                alpha=0.15, s=12, color=cat_col)
    ax3.plot([-1, 1], [-1, 1], color='grey',
             linewidth=1, linestyle='--', label='y = x  (no weighting effect)')
    ax3.set_xlim(-1.5, 1.5)
    ax3.set_ylim(-1.5, 1.5)
    ax3.set_title('C.  Raw vs Weighted Sentiment', loc='left',
                  fontsize=11, fontweight='bold')
    ax3.set_xlabel('Raw Sentiment Score', fontsize=10)
    ax3.set_ylabel('Engagement-Weighted Score', fontsize=10)
    ax3.legend(fontsize=8)
    ax3.text(0.03, 0.97,
             'Points below the line =\nhigh-engagement negative comments\n(silent majority signal)',
             transform=ax3.transAxes, fontsize=7.5, va='top',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                       edgecolor='#ccc', alpha=0.9))

    # ── Panel 4 (bottom, full width): Decay Curve ────────────────────────────
    ax4 = fig.add_subplot(gs[2, :])
    ax4.bar(daily['comment_date'], daily['volume'],
            color=cat_col, alpha=0.18, width=1, label='Daily Volume')

    if decay:
        ax4.plot(decay['fit_dates'], decay['fit_values'],
                 color='#E74C3C', linewidth=2.5, zorder=5,
                 label=(f"Decay Fit  "
                        f"t½ = {decay['half_life']:.1f} d  |  "
                        f"λ = {decay['lam']:.4f}  |  "
                        f"Dynamic window = {decay['fit_days']} d"))
        ax4.axhline(decay['baseline_vol'], color='#E74C3C',
                    linewidth=1, linestyle=':', alpha=0.5,
                    label=f"Pre-event baseline ({decay['baseline_vol']:.0f} comments/day)")
        ax4.axvline(decay['peak_date'], color='#F39C12',
                    linewidth=1.2, linestyle='--', alpha=0.7,
                    label=f"Peak: {decay['peak_date'].date()}")

        # Developer taxonomy annotation
        taxonomy = ('🔥 "Wait it Out"' if decay['half_life'] <= 14
                    else '⚠️  "Fundamental Damage"')
        ax4.text(0.98, 0.96, taxonomy,
                 transform=ax4.transAxes, fontsize=10, fontweight='bold',
                 va='top', ha='right', color='white',
                 bbox=dict(boxstyle='round,pad=0.5',
                           facecolor='#27AE60' if decay['half_life'] <= 14
                           else '#C0392B',
                           alpha=0.88))
    else:
        ax4.text(0.5, 0.6, 'Insufficient data for decay fit\nin this time range.',
                 transform=ax4.transAxes, ha='center', va='center',
                 fontsize=11, color='grey')

    ax4.set_title('D.  Exponential Volume Decay', loc='left',
                  fontsize=11, fontweight='bold')
    ax4.set_ylabel('Comment Volume', fontsize=10)
    ax4.set_xlabel('Date', fontsize=10)
    ax4.legend(fontsize=8, loc='upper right', framealpha=0.9)

    plt.show()

# ── Widgets ───────────────────────────────────────────────────────────────────

cat_dropdown = widgets.Dropdown(
    options      = CATEGORY_OPTIONS,
    value        = 'Game_Related',
    description  = 'Category:',
    style        = {'description_width': '80px'},
    layout       = widgets.Layout(width='280px'),
)

time_dropdown = widgets.Dropdown(
    options     = list(PRESET_RANGES.keys()),
    value       = 'All Time',
    description = 'Time Range:',
    style       = {'description_width': '80px'},
    layout      = widgets.Layout(width='200px'),
)

run_btn = widgets.Button(
    description  = '▶  Update Dashboard',
    button_style = 'primary',
    layout       = widgets.Layout(width='180px', height='34px'),
)

ai_note = widgets.HTML(
    value  = '',
    layout = widgets.Layout(margin='0 0 0 12px'),
)

out = widgets.Output()

def _on_cat_change(change):
    if change['new'] == 'AI_Voice_Related':
        ai_note.value = (
            '<span style="color:#8E44AD; font-size:12px;">'
            '⚠️  AI Voice: data filtered from Nov 12, 2025 onward</span>'
        )
    else:
        ai_note.value = ''

def _on_run(_):
    with out:
        clear_output(wait=True)
        _draw_dashboard(cat_dropdown.value, time_dropdown.value)

cat_dropdown.observe(_on_cat_change, names='value')
run_btn.on_click(_on_run)

controls = widgets.HBox(
    [cat_dropdown, time_dropdown, run_btn, ai_note],
    layout=widgets.Layout(
        align_items='center',
        padding='10px',
        border='1px solid #ddd',
        border_radius='6px',
        margin='0 0 12px 0',
    ),
)

display(controls, out)

# Draw once on load with defaults
_on_run(None)

Output()

End of notebook.